# 01 — What is a World Model?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Mattral/rssmlite/blob/main/notebooks/01_world_model_intro.ipynb)

A conceptual walkthrough of the RSSM (Recurrent State-Space Model) — what it
learns, how it represents state, and how to read its outputs. No long training
run here; everything uses a randomly-initialised model so you see the API
shape before any real training happens.

**Runtime:** CPU is fine for this notebook. No GPU needed.


In [ ]:
# ── Bootstrap ─────────────────────────────────────────────────────────────
!pip install -q rssmlite[envs,viz]

import torch, gymnasium as gym, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
print("rssmlite ready")


## The two-part state

At every timestep the RSSM maintains:
- **`deter`** — a deterministic GRU hidden state (smooth gradient-friendly memory)
- **`stoch`** — a stochastic categorical latent (32 groups × 32 classes by default)

`feature = concat(deter, flatten(stoch))` is what the decoder, reward head,
continue head, actor, and critic all read.


In [ ]:
from rssmlite import RSSM

# Small dims for fast CPU demo
rssm = RSSM(obs_dim=4, action_dim=2, deter_dim=64,
            embed_dim=64, num_categoricals=8, num_classes=8, hidden_dim=64)

state = rssm.initial_state(batch_size=1, device=torch.device("cpu"))
print("deter shape :", state["deter"].shape)   # (1, 64)
print("stoch shape :", state["stoch"].shape)   # (1, 64)  = 8*8 flattened
print("feature dim :", rssm.feature_dim)       # 64+64 = 128


## Observe: teacher-forced pass over a real sequence

`rssm.observe(obs_seq, action_seq)` runs the recurrence over a batch of real
transitions, computing both the **posterior** (which sees the real observation)
and the **prior** (which doesn't) at every step. The gap between them is what
the KL loss closes during training.


In [ ]:
B, T = 2, 20   # batch size, sequence length

obs_seq    = torch.randn(B, T, 4)   # fake CartPole observations
action_seq = torch.randn(B, T, 2)   # fake actions (one-hot in real use)

rollout = rssm.observe(obs_seq, action_seq)

print("Keys returned:", list(rollout.keys()))
print("deter      :", rollout["deter"].shape)        # (B, T, 64)
print("stoch      :", rollout["stoch"].shape)        # (B, T, 64)
print("post_logits:", rollout["post_logits"].shape)  # (B, T, 8, 8)
print("prior_logits:", rollout["prior_logits"].shape)


## The stochastic latent is always one-hot per group

`stoch` has shape `(B, T, num_categoricals * num_classes)`. When reshaped to
`(B, T, num_categoricals, num_classes)`, each of the 8 groups sums to exactly
1 — it's a discrete, hard assignment, not a soft probability.

This is what makes the representation sharp: each group "votes" for one class
and nothing else. Compare to a Gaussian VAE, where every latent dim gets a
fuzzy real-valued activation.


In [ ]:
stoch_grouped = rollout["stoch"].view(B, T, 8, 8)
group_sums = stoch_grouped.sum(dim=-1)   # should all be 1.0

print("Min group sum:", group_sums.min().item())   # 1.0
print("Max group sum:", group_sums.max().item())   # 1.0

# Visualise the one-hot structure for one timestep
fig, axes = plt.subplots(1, 4, figsize=(12, 2))
for i, ax in enumerate(axes):
    ax.bar(range(8), stoch_grouped[0, 0, i].detach().numpy())
    ax.set_title(f"group {i}")
    ax.set_ylim(0, 1.2)
    ax.set_xticks([])
plt.suptitle("Stochastic latent: first 4 groups at t=0 (random init)", y=1.02)
plt.tight_layout()
plt.savefig("latent_groups.png", bbox_inches="tight")
plt.show()
print("Each bar is one-hot — exactly one class is active per group.")


## Imagine: rolling forward without observations

`rssm.imagine(initial_state, policy, horizon)` uses only the **prior** — no
real observations. This is what the actor-critic is trained on: the model
predicts what will happen if the policy takes a given sequence of actions,
without ever touching the real environment.


In [ ]:
def random_policy(feature):
    return torch.randn(feature.shape[0], 2)   # random 2-dim action

horizon = 15
imagined = rssm.imagine(state, random_policy, horizon)

print("Imagined deter:", imagined["deter"].shape)   # (1, 15, 64)
print("Imagined stoch:", imagined["stoch"].shape)
print("Imagined action:", imagined["action"].shape)

# Decode imagined states back to observation predictions
feature_seq = torch.cat([imagined["deter"], imagined["stoch"]], dim=-1)
obs_pred = rssm.decoder(feature_seq)
print("\nPredicted observations (untrained, random noise):", obs_pred.shape)


## The world model loss

`rssm.loss()` combines four terms. Let's look at what each contributes on a
randomly-initialised model — before training the losses are high, especially KL.


In [ ]:
reward_seq  = torch.randn(B, T)
continue_seq = torch.ones(B, T)

losses = rssm.loss(obs_seq, action_seq, reward_seq, continue_seq)

print("Loss breakdown (randomly initialised model):")
for k, v in losses.items():
    print(f"  {k:10s}: {v.item():.4f}")

print("\nNote: KL is at the free-bits floor (1.0) — the model gets that")
print("much representational budget without penalty. As training proceeds")
print("the posterior and prior will diverge more, then converge as the")
print("prior learns to track the posterior.")


## What's next

This notebook showed the API shape on a randomly-initialised model.
`02_cartpole_experiment.ipynb` runs a real training loop on CartPole-v1
(~20 min on a free Colab T4) and plots training curves, reconstruction
quality, and imagined rollouts from a trained model.
